## watsonx.ai Runtimeを使用したDecision Optimizationモデルのデプロイ

このノートブックでは、watsonx.ai Python Clientを使用してDecision Optimizationモデルをデプロイし、ジョブを作成・監視し、ソリューションを取得する方法を示します。

このノートブックはPythonで実行されます。

**目次:**

- [watsonx.aiクライアントのセットアップ](#setup)
- [クライアントインスタンスの作成](#create)
- [モデルアーカイブの準備](#prepare)
- [watsonx.ai Runtimeへのモデルのアップロード](#upload)
- [デプロイメントの作成](#deploy)
- [デプロイされたモデルのインラインデータを使用したジョブの作成と監視](#job)
- [ソリューションの表示](#display)
- [同じデプロイメントを使用した別の問題の解決](#problem)
- [まとめ](#summary)

<a id='setup'></a>
### watsonx.aiクライアントのセットアップ

このノートブックのサンプルコードを使用する前に、以下を行う必要があります：

- <a href="https://cloud.ibm.com/catalog?category=ai" target="_blank" rel="noopener noreferrer">watsonx.ai Runtime Service</a>インスタンスを作成します。無料プランが提供されており、インスタンスの作成方法については<a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas" target="_blank" rel="noopener noreferrer">こちら</a>で確認できます。


watsonx.aiクライアントライブラリをインポートします。

### output.zipのダウンロードとsolution.csvの表示

ジョブの出力からoutput.zipをダウンロードし、その中のsolution.csvを表示します。

In [1]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials
import pandas as pd
from dotenv import load_dotenv
import os

# .envファイルから環境変数を読み込む
load_dotenv()

True

<a id='create'></a>
### クライアントインスタンスの作成

IBM Cloud APIキーを使用します。APIキーの取得方法は<a href="https://dataplatform.cloud.ibm.com/docs/content/DO/WML_Deployment/DeployModelRest.html?audience=wdp&context=cpdaas#tasktask_deploymodelREST__prereq_el2_nft_bhb">こちら</a>、インスタンスURLは<a href="https://cloud.ibm.com/apidocs/machine-learning#endpoint-url">こちら</a>で確認できます。

In [2]:
# ========================================
# 設定: 認証情報（環境変数から読み込み）
# ========================================
# .envファイルに以下の環境変数を設定してください：
# WATSONX_API_KEY=your_api_key_here
# WATSONX_INSTANCE_URL=https://jp-tok.ml.cloud.ibm.com
# WATSONX_SPACE_ID=your_space_id_here

API_KEY = os.getenv('WATSONX_API_KEY')  # IBM Cloud APIキー
INSTANCE_URL = os.getenv('WATSONX_INSTANCE_URL')  # watsonx.ai インスタンスURL

# 環境変数が設定されているか確認
if not API_KEY or not INSTANCE_URL:
    raise ValueError(
        "環境変数が設定されていません。\n"
        ".envファイルにWATSONX_API_KEYとWATSONX_INSTANCE_URLを設定してください。"
    )

# 認証情報を使用してクライアントをインスタンス化
credentials = Credentials(
    api_key=API_KEY,
    url=INSTANCE_URL
)

client = APIClient(credentials)

In [3]:
# ========================================
# 設定: スペース情報（環境変数から読み込み）
# ========================================
# .envファイルのWATSONX_SPACE_IDから読み込み
SPACE_ID = os.getenv('WATSONX_SPACE_ID')  # スペースID（必須）

# 環境変数が設定されているか確認
if not SPACE_ID:
    raise ValueError(
        "環境変数WATSONX_SPACE_IDが設定されていません。\n"
        ".envファイルにWATSONX_SPACE_IDを設定してください。"
    )


# スペースIDを指定してクライアントを再作成
client = APIClient(credentials, space_id=SPACE_ID)

<a id='prepare'></a>
### モデルアーカイブの準備

model.pyファイルをサブディレクトリに配置し、tar.gzファイルを作成します。モデルは2つの部分で構成されています：
* ファイルから`inputs`ディクショナリを作成し、`outputs`ディクショナリからファイルを作成する関数群
* inputsとoutputsディクショナリを使用する最適化モデル

`write_file`コマンドを使用してモデルを`main.py`ファイルに書き込みます。

`tar`コマンドを使用してtarアーカイブを作成します。

In [4]:
import tarfile
def reset(tarinfo):
    tarinfo.uid = tarinfo.gid = 0
    tarinfo.uname = tarinfo.gname = "root"
    return tarinfo
tar = tarfile.open("model.tar.gz", "w:gz")
tar.add("model/main.py", arcname="main.py", filter=reset)
tar.close()

<a id='upload'></a>
### watsonx.ai Runtimeへのモデルのアップロード

以下を使用してwatsonx.ai Runtimeにモデルを保存します：
* 前に作成したtarアーカイブ
* モデルタイプとランタイムを含むメタデータ

`model_uid`を取得します。

In [5]:
# ========================================
# 設定: モデル情報
# ========================================
MODEL_NAME = "Diet"  # モデル名
MODEL_DESCRIPTION = "Model for Diet"  # モデルの説明
MODEL_TYPE = "do-docplex_22.1"  # モデルタイプ
SOFTWARE_SPEC_NAME = "do_22.1"  # ソフトウェア仕様名
MODEL_ARCHIVE_PATH = "./model.tar.gz"  # モデルアーカイブのパス

# モデルのメタデータを定義
domodel_metadata = {
    client.repository.ModelMetaNames.NAME: MODEL_NAME,
    client.repository.ModelMetaNames.DESCRIPTION: MODEL_DESCRIPTION,
    client.repository.ModelMetaNames.TYPE: MODEL_TYPE,
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: client.software_specifications.get_id_by_name(SOFTWARE_SPEC_NAME)
}

# モデルをwatsonx.aiにアップロード
model_details = client.repository.store_model(model=MODEL_ARCHIVE_PATH, meta_props=domodel_metadata)

# モデルのユニークIDを取得（後のデプロイメントで使用）
model_uid = client.repository.get_model_id(model_details)

# 必要に応じてモデルuidを出力
# print(f"Model UID: {model_uid}")

<a id='deploy'></a>
### デプロイメントの作成

以下の情報を提供してモデルのバッチデプロイメントを作成します：
* コンピュートノードの最大数
* コンピュートノードのTシャツサイズ

`deployment_uid`を取得します。

In [6]:
# ========================================
# 設定: デプロイメント情報
# ========================================
DEPLOYMENT_NAME = "Diet Deployment"  # デプロイメント名
DEPLOYMENT_DESCRIPTION = "Diet Deployment"  # デプロイメントの説明
HARDWARE_SIZE = "S"  # ハードウェアサイズ（S, M, L, XL）
NUM_NODES = 1  # 使用するノード数

# デプロイメントのメタデータを定義
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: DEPLOYMENT_NAME,
    client.deployments.ConfigurationMetaNames.DESCRIPTION: DEPLOYMENT_DESCRIPTION,
    client.deployments.ConfigurationMetaNames.BATCH: {},
    client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {'name': HARDWARE_SIZE, 'num_nodes': NUM_NODES}
}

# デプロイメントを作成
deployment_details = client.deployments.create(model_uid, meta_props=meta_props)

# デプロイメントのユニークIDを取得（後のジョブ実行で使用）
deployment_uid = client.deployments.get_id(deployment_details)

# 必要に応じてデプロイメントIDを出力
# print(f"Deployment UID: {deployment_uid}")



######################################################################################

Synchronous deployment creation for id: '88d255ee-86b6-407d-85e4-7ba6f1bd739e' started

######################################################################################


ready.


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='e6d333bd-79fc-4648-aa1d-df5cafc25431'
-----------------------------------------------------------------------------------------------




<a id='job'></a>
### デプロイされたモデルのインラインデータを使用したジョブの作成と監視

インライン入力データを含むペイロードを作成します。

このペイロードとデプロイメントを使用して新しいジョブを作成します。

`job_uid`を取得します。

解決時に中間解を有効にしたい場合は、``oaas.outputUploadPeriod``解決パラメータ（分単位で表現）のコメントを外してください。

各中間解でログを取得したい場合は、``oaas.outputUploadFiles``のコメントも外すことができます。

In [7]:
# ========================================
# 設定: 入力データ（1回目のジョブ実行）
# ========================================
INPUT_DATA_NAME = "input.zip"  # データアセット名
INPUT_DATA_PATH = "./model/input.zip"  # ローカルファイルパス（圧縮された入力ファイル）

# 入力データをwatsonx.aiにアップロード
da_details = client.data_assets.create(
    name=INPUT_DATA_NAME,
    file_path=INPUT_DATA_PATH
)

# アップロードされたデータアセットのIDを取得
# このIDは後でジョブ実行時に入力ファイルとして参照されます
input_da_id = da_details["metadata"]["asset_id"]

Creating data asset...
SUCCESS


In [8]:
# ========================================
# ジョブペイロードの作成（1回目のジョブ実行）
# ========================================
solve_payload = {
    # ソルバーパラメータの設定
    "solve_parameters": {
        # 中間解のアップロード間隔（分単位）- 必要に応じてコメントを外す
        # "oaas.outputUploadPeriod": "1",
        # 中間解でアップロードするファイルパターン - 必要に応じてコメントを外す
        # "oaas.outputUploadFiles": ".*\\.txt",
        # CPLEXソルバーのログ出力を有効化（デバッグや進捗確認に有用）
        "oaas.logTailEnabled": "true",
        # CPLEXソルバーログ全体をジョブ出力のアセットとして保存する
        "oaas.logAttachmentName": "log.txt"
    },
    # 入力データの指定
    client.deployments.DecisionOptimizationMetaNames.INPUT_DATA_REFERENCES: [
        {
            "id": "input.zip",  # アップロードした入力ファイル名
            "type": "data_asset",  # データアセットとして参照
            "location": {
                "href": f"/v2/assets/{input_da_id}?space_id={SPACE_ID}"
            }
        }
    ],
    # インライン出力データの指定（KPIと統計情報）
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA: [
        {"id": "kpis.csv"},  # KPI（主要業績評価指標）
        {"id": "stats.csv"}  # 統計情報
    ],
    # データアセットとして保存する出力ファイルの指定
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA_REFERENCES: [
        {
            "id": "output.zip",  # 結果出力ファイルの識別子
            "type": "data_asset",  # データアセットとして保存
            "location": {
                "name": "output.zip"  # 新しいデータアセットとして作成される
            }
        },
        {
            "id": "log.txt",  # ログ出力ファイルの識別子
            "type": "data_asset",  # データアセットとして保存
            "location": {
                "name": "log.txt"  # 新しいデータアセットとして作成される
            }
        }
    ]
}

# ジョブを作成して実行
job_details = client.deployments.create_job(deployment_uid, solve_payload)
job_uid = client.deployments.get_job_id(job_details)

# 必要に応じてジョブIDを出力
# print(f"Job UID: {job_uid}")

ジョブが完了するまでジョブステータスを表示します。

新しいデプロイメントの最初のジョブは、コンピュートノードを起動する必要があるため、時間がかかる場合があります。

In [9]:
# ========================================
# 設定: ジョブ監視パラメータ
# ========================================
POLL_INTERVAL_SECONDS = 5  # ジョブ状態をチェックする間隔（秒）
SEPARATOR_LINE = "=" * 60  # 区切り線

import time

# ジョブの状態を監視（完了するまでループ）
print(SEPARATOR_LINE)
print("ジョブの監視を開始")
print(f"状態の更新は{POLL_INTERVAL_SECONDS}秒ごとに行われます")
print(SEPARATOR_LINE)
print()

while True:
    # ジョブの詳細情報を取得
    job_status = client.deployments.get_job_details(job_uid)
    
    # 現在の状態を取得
    state = job_status['entity']['decision_optimization']['status']['state']
    
    # 状態を表示
    print(f"[状態] ジョブ状態: {state}")
    
    # 完了状態をチェック
    if state in ["completed", "failed", "canceled"]:
        print()
        print(SEPARATOR_LINE)
        if state == "completed":
            print("[完了] ジョブが正常に完了しました")
        elif state == "failed":
            print("[失敗] ジョブが失敗しました")
        elif state == "canceled":
            print("[中止] ジョブがキャンセルされました")
        print(SEPARATOR_LINE)
        print()
        break
    
    # CPLEXの解法状態が利用可能な場合は表示
    if 'solve_state' in job_status['entity']['decision_optimization']:
        solve_state = job_status['entity']['decision_optimization']['solve_state']
        print(f"  [詳細] 解法状態: {solve_state}")
    
    # 指定された間隔で待機
    time.sleep(POLL_INTERVAL_SECONDS)

ジョブの監視を開始
状態の更新は5秒ごとに行われます

[状態] ジョブ状態: completed

[完了] ジョブが正常に完了しました



In [10]:
# 結果表示
import pprint
job_details=client.deployments.get_job_details(job_uid)
pprint.pprint(job_details)

{'entity': {'decision_optimization': {'input_data_references': [{'connection': {},
                                                                 'id': 'input.zip',
                                                                 'location': {'href': '/v2/assets/ca5f8962-4a88-47ad-81e6-29ea13a74fbc?space_id=e4d818d5-b279-4f60-bbb2-bb122d2d5890'},
                                                                 'type': 'data_asset'}],
                                      'output_data': [{'fields': ['Name',
                                                                  'Value'],
                                                       'id': 'kpis.csv',
                                                       'values': [['Total 食物繊維',
                                                                   25],
                                                                  ['Total '
                                                                   'タンパク質',
                                

<a id='display'></a>
### ソリューションの抽出と表示

出力ソリューションを表示します。

KPI Total Caloriesの値を表示します。

In [11]:
# kpiのデータフレームを作成
kpis = pd.DataFrame(job_details['entity']['decision_optimization']['output_data'][0]['values'], 
                        columns = job_details['entity']['decision_optimization']['output_data'][0]['fields'])
kpis

,Name,Value
0,Total 食物繊維,25.000000
1,Total タンパク質,51.173722
2,PROGRESS_CURRENT_OBJECTIVE,2.690409
3,Total ビタミンA,8518.432542
4,Total 鉄分,11.278318
5,Total カルシウム,800.000000
6,Total カロリー,2000.000000
7,Minimal cost,2.690409
8,Total 炭水化物,256.805764


In [13]:
# statsのデータフレームを作成
stats = pd.DataFrame(job_details['entity']['decision_optimization']['output_data'][1]['values'], 
                        columns = job_details['entity']['decision_optimization']['output_data'][1]['fields'])
stats

,Name,Value
0,cplex.size.integerVariables,0
1,cplex.size.linearConstraints,7
2,cplex.modelType,LP
3,cplex.size.semiintegerVariables,0
4,cplex.size.variables,9
5,cplex.size.continousVariables,9
6,job.inputsReadMs,38448
7,cplex.size.booleanVariables,0
8,job.coresCount,1
9,cplex.size.semicontinuousVariables,0


In [14]:
# ========================================
# 1回目のジョブ実行結果の取得と表示
# ========================================
# output.zipをダウンロードして解凍し、solution.csvを表示
import zipfile
import io

# ジョブの最終詳細情報を取得
job_details = client.deployments.get_job_details(job_uid)

# output_data_referencesを確認（データアセットとして保存された出力）
print("=== 出力データ参照の確認 ===")
if 'output_data_references' in job_details['entity']['decision_optimization']:
   
    # output.zipのデータアセットIDを取得
    output_zip_asset_id = None
    for ref in job_details['entity']['decision_optimization']['output_data_references']:
        if ref.get('id') == 'output.zip':
            if 'location' in ref and 'id' in ref['location']:
                output_zip_asset_id = ref['location']['id']
                print(f"\noutput.zipのアセットIDが見つかりました: {output_zip_asset_id}")
                break
    
    if output_zip_asset_id:
        # データアセットからoutput.zipをダウンロード
        print("\noutput.zipをダウンロード中...")
        output_zip_path = client.data_assets.download(
            asset_id=output_zip_asset_id,
            filename="output.zip"
        )
        print(f"ダウンロード完了: {output_zip_path}")
        
        # ZIPファイルを解凍してsolution.csvを表示
        print("\nZIPファイルを解凍中...")
        with zipfile.ZipFile(output_zip_path, 'r') as zip_ref:
            print("\nZIPファイルの内容:")
            for file_name in zip_ref.namelist():
                print(f"  - {file_name}")
            
            # solution.csvを読み込む
            if 'solution.csv' in zip_ref.namelist():
                with zip_ref.open('solution.csv') as csv_file:
                    solution_from_zip = pd.read_csv(csv_file)
                    print("\n=== output.zipから取得したsolution.csv ===\n")
                    display(solution_from_zip)
            else:
                print("\nsolution.csvがZIPファイル内に見つかりませんでした。")
    else:
        print("\noutput.zipのアセットIDが見つかりませんでした。")
else:
    print("\noutput_data_referencesが見つかりませんでした。")



=== 出力データ参照の確認 ===

output.zipのアセットIDが見つかりました: 120ff431-734c-4fd9-97a1-616bdc0305ca

output.zipをダウンロード中...
Successfully saved data asset content to file: 'output.zip'
ダウンロード完了: C:\_onedrive\OneDrive - IBM\01code\2026\260114docplexWML\output.zip

ZIPファイルを解凍中...

ZIPファイルの内容:
  - solution.csv

=== output.zipから取得したsolution.csv ===



,Food,value
0,スパゲッティ（ソース付き）,2.155172
1,チョコチップクッキー,10.000000
2,低脂肪牛乳,1.831167
3,ホットドッグ,0.929698


### log.txtのダウンロード

ジョブの実行ログをダウンロードします。

In [15]:
# ========================================
# 設定: ログ表示パラメータ
# ========================================
MAX_LOG_LINES = 50  # 表示するログの最大行数

# log.txtのデータアセットIDを取得してダウンロード
log_txt_asset_id = None
if 'output_data_references' in job_details['entity']['decision_optimization']:
    for ref in job_details['entity']['decision_optimization']['output_data_references']:
        if ref.get('id') == 'log.txt':
            if 'location' in ref and 'id' in ref['location']:
                log_txt_asset_id = ref['location']['id']
                print(f"log.txtのアセットIDが見つかりました: {log_txt_asset_id}")
                break

if log_txt_asset_id:
    # データアセットからlog.txtをダウンロード
    print("\nlog.txtをダウンロード中...")
    log_txt_path = client.data_assets.download(
        asset_id=log_txt_asset_id,
        filename="log.txt"
    )
    print(f"ダウンロード完了: {log_txt_path}")
    
    # ログファイルの内容を表示（最初のMAX_LOG_LINES行）
    print(f"\n=== log.txtの内容（最初の{MAX_LOG_LINES}行） ===\n")
    with open(log_txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for i, line in enumerate(lines[:MAX_LOG_LINES], 1):
            print(f"{i}: {line}", end='')
        if len(lines) > MAX_LOG_LINES:
            print(f"\n... (残り {len(lines) - MAX_LOG_LINES} 行)")
else:
    print("\nlog.txtのアセットIDが見つかりませんでした。")

log.txtのアセットIDが見つかりました: 442c0e4c-f26d-46cd-822a-6a97df18e816

log.txtをダウンロード中...
Successfully saved data asset content to file: 'log.txt'
ダウンロード完了: C:\_onedrive\OneDrive - IBM\01code\2026\260114docplexWML\log.txt

=== log.txtの内容（最初の50行） ===

1: [2026-01-20T08:53:18.112Z, WARNING] DO Runtime 24.1 on Python 3.11 is used as default with pandas 2.1.1 libraries.
2: [2026-01-20T08:53:18.810Z, INFO] Loaded dtype schemas: {'diet_food_nutrients.csv': {'Food': 'string', 'カロリー': 'float64', 'カルシウム': 'float64', '鉄分': 'float64', 'ビタミンA': 'float64', '食物繊維': 'float64', '炭水化物': 'float64', 'タンパク質': 'float64'}, 'diet_food.csv': {'name': 'string', 'unit_cost': 'float64', 'qmin': 'float64', 'qmax': 'float64'}, 'diet_nutrients.csv': {'name': 'string', 'qmin': 'float64', 'qmax': 'float64'}}
3: [2026-01-20T08:53:18.862Z, INFO] Model: diet
4: [2026-01-20T08:53:18.863Z, INFO]  - number of variables: 9
5: [2026-01-20T08:53:18.863Z, INFO]    - binary=0, integer=0, continuous=9
6: [2026-01-20T08:53:18.864Z, INFO] 

### リソースのクリーンアップ（1回目のジョブ実行後）

ダウンロード後、不要になったデータアセットとジョブを削除します。
これにより、watsonx.aiのストレージ容量を節約できます。

In [16]:
# 1. 出力データアセットの削除
client.data_assets.delete(asset_id=output_zip_asset_id)
client.data_assets.delete(asset_id=log_txt_asset_id)
print(f"[OK] [1/3] 出力データアセットを削除しました")
print(f"        Asset ID: {output_zip_asset_id}")
print(f"        Asset ID: {log_txt_asset_id}")

[OK] [1/3] 出力データアセットを削除しました
        Asset ID: 120ff431-734c-4fd9-97a1-616bdc0305ca
        Asset ID: 442c0e4c-f26d-46cd-822a-6a97df18e816


In [17]:
# 2. ジョブの削除 実際には削除できない。
client.deployments.delete_job(job_uid)
print(f"[OK] [2/5] ジョブを削除しました")
print(f"        Job ID: {job_uid}")

[OK] [2/5] ジョブを削除しました
        Job ID: 4baa6606-2db0-4e94-9828-c95190d7f56a


In [18]:
# 3. 入力データアセットinput.zip（入力データファイル）の削除
client.data_assets.delete(input_da_id)
print(f"[OK] [3/5] 入力データアセットを削除しました")
print(f"        Asset ID: {input_da_id}")

[OK] [3/5] 入力データアセットを削除しました
        Asset ID: ca5f8962-4a88-47ad-81e6-29ea13a74fbc


<a id='problem'></a>
###  同じデプロイメントを使用した別の問題の解決

変更された入力データで新しいペイロードを作成します。

In [19]:
# ========================================
# 設定: 入力データ（2回目のジョブ実行）
# ========================================
# 注意: これは2回目のジョブ実行用の入力データです
INPUT_DATA_NAME_2 = "input.zip"  # データアセット名
INPUT_DATA_PATH_2 = "./model/input2.zip"  # ローカルファイルパス（異なる入力データ）

# 2回目の入力データをwatsonx.aiにアップロード
da_details = client.data_assets.create(
    name=INPUT_DATA_NAME_2,
    file_path=INPUT_DATA_PATH_2
)

# アップロードされたデータアセットのIDを取得
# このIDは後でジョブ実行時に入力ファイルとして参照されます
input_da_id = da_details["metadata"]["asset_id"]

Creating data asset...
SUCCESS


In [20]:
# ========================================
# ジョブペイロードの作成（2回目のジョブ実行）
# ========================================
# 注意: これは2回目のジョブ実行用のペイロードです（同じデプロイメントを再利用）
solve_payload = {
    # ソルバーパラメータの設定
    "solve_parameters": {
        # 中間解のアップロード間隔（分単位）- 必要に応じてコメントを外す
        # "oaas.outputUploadPeriod": "1",
        # 中間解でアップロードするファイルパターン - 必要に応じてコメントを外す
        # "oaas.outputUploadFiles": ".*\\.txt",
        # CPLEXソルバーのログ出力を有効化（デバッグや進捗確認に有用）
        "oaas.logTailEnabled": "true",
        # CPLEXソルバーログ全体をジョブ出力のアセットとして保存する
        "oaas.logAttachmentName": "log.txt"
    },
    # 入力データの指定
    client.deployments.DecisionOptimizationMetaNames.INPUT_DATA_REFERENCES: [
        {
            "id": "input.zip",  # アップロードした入力ファイル名
            "type": "data_asset",  # データアセットとして参照
            "location": {
                "href": f"/v2/assets/{input_da_id}?space_id={SPACE_ID}"
            }
        }
    ],
    # インライン出力データの指定（KPIと統計情報）
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA: [
        {"id": "kpis.csv"},  # KPI（主要業績評価指標）
        {"id": "stats.csv"}  # 統計情報
    ],
    # データアセットとして保存する出力ファイルの指定
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA_REFERENCES: [
        {
            "id": "output.zip",  # 結果出力ファイルの識別子
            "type": "data_asset",  # データアセットとして保存
            "location": {
                "name": "output.zip"  # 新しいデータアセットとして作成される
            }
        },
        {
            "id": "log.txt",  # ログ出力ファイルの識別子
            "type": "data_asset",  # データアセットとして保存
            "location": {
                "name": "log.txt"  # 新しいデータアセットとして作成される
            }
        }
    ]
}

# ジョブを作成して実行（2回目）
job_details = client.deployments.create_job(deployment_uid, solve_payload)
job_uid = client.deployments.get_job_id(job_details)

# 必要に応じてジョブIDを出力
# print(f"Job UID: {job_uid}")

新しいジョブを作成します。

ジョブが完了するまでジョブステータスを表示します。

In [21]:
while job_details['entity']['decision_optimization']['status']['state'] not in ['completed', 'failed', 'canceled']:
    print(job_details['entity']['decision_optimization']['status']['state'] + '...')
    time.sleep(5)
    job_details=client.deployments.get_job_details(job_uid)

print( job_details['entity']['decision_optimization']['status']['state'])

queued...
completed


In [22]:
# ジョブの最終詳細情報を取得
job_details = client.deployments.get_job_details(job_uid)

# 出力データアセットのIDを取得
output_zip_asset_id = job_details['entity']['decision_optimization']["output_data_references"][0]["location"]["id"]
# solution.xmlをダウンロード
client.data_assets.download(
    asset_id=output_zip_asset_id,  # 出力データアセットのID
    filename="output.zip"     # 保存するファイル名
)
with zipfile.ZipFile(output_zip_path, 'r') as zip_ref:
    # solution.csvを読み込む
    with zip_ref.open('solution.csv') as csv_file:
        solution_from_zip = pd.read_csv(csv_file)
        print("\n=== output.zipから取得したsolution.csv ===\n")
        display(solution_from_zip)

Successfully saved data asset content to file: 'output.zip'

=== output.zipから取得したsolution.csv ===



,Food,value
0,ローストチキン,0.267123
1,スパゲッティ（ソース付き）,2.106841
2,チョコチップクッキー,5.299290
3,低脂肪牛乳,1.990285
4,レーズンブラン,0.140161


In [24]:
# ジョブの最終詳細情報を取得
job_details = client.deployments.get_job_details(job_uid)

# 出力ログアセットのIDを取得
log_txt_asset_id = job_details['entity']['decision_optimization']["output_data_references"][1]["location"]["id"]


### デプロイメントの削除

デプロイメントを削除するには、以下のメソッドを使用します。

In [25]:
# 1. 出力データアセットの削除
client.data_assets.delete(asset_id=output_zip_asset_id)
client.data_assets.delete(asset_id=log_txt_asset_id)
print(f"[OK] [1/5] 出力データアセットを削除しました")
print(f"        Asset ID: {output_zip_asset_id}")
print(f"        Asset ID: {log_txt_asset_id}")


[OK] [1/5] 出力データアセットを削除しました
        Asset ID: bca24433-06ca-415b-bcef-f6c4a0d91964
        Asset ID: a88d241f-aef7-49b7-867f-f5d712250e65


In [26]:
# 2. ジョブの削除 実際には削除できない。
client.deployments.delete_job(job_uid)
print(f"[OK] [2/5] ジョブを削除しました")
print(f"        Job ID: {job_uid}")


[OK] [2/5] ジョブを削除しました
        Job ID: 301536a9-d561-4abe-865b-e89d462d46a0


In [27]:
# 3. 入力データアセットinput.zip（入力データファイル）の削除
client.data_assets.delete(input_da_id)
print(f"[OK] [3/5] 入力データアセットを削除しました")
print(f"        Asset ID: {input_da_id}")


[OK] [3/5] 入力データアセットを削除しました
        Asset ID: 03d192b9-dd6b-4c90-9de7-c1f7d49c4a14


In [28]:
# 4. デプロイメントの削除
client.deployments.delete(deployment_uid)
print(f"[OK] [4/5] デプロイメントを削除しました")
print(f"        Deployment ID: {deployment_uid}")


[OK] [4/5] デプロイメントを削除しました
        Deployment ID: e6d333bd-79fc-4648-aa1d-df5cafc25431


In [29]:
# 5. モデルの削除
client.repository.delete(model_uid)
print(f"[OK] [5/5] モデルを削除しました")
print(f"        Model ID: {model_uid}")


[OK] [5/5] モデルを削除しました
        Model ID: 88d255ee-86b6-407d-85e4-7ba6f1bd739e


<a id='summary'></a>
### まとめと次のステップ

このノートブックが正常に完了しました！

学習した内容：

- watsonx.aiクライアントの使用方法
- モデルアーカイブの準備とwatsonx.ai Runtimeへのモデルのアップロード
- デプロイメントの作成
- デプロイされたモデルのインラインデータを使用したジョブの作成と監視
- ソリューションの表示

より多くのサンプル、チュートリアル、ドキュメントについては、オンラインドキュメントをご確認ください：
* <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=cpdaas" target="_blank" rel="noopener noreferrer">IBM Cloud Pak for Data as a Serviceドキュメント</a>
* <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx" target="_blank" rel="noopener noreferrer">IBM watsonx.aiドキュメント</a>

In [31]:
# バージョン情報を表示
import sys
import ibm_watsonx_ai
print(f"Python version: {sys.version}")
print(f"ibm-watsonx-ai version: {ibm_watsonx_ai.__version__}")

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
ibm-watsonx-ai version: 1.4.11


<hr>
Copyright © 2019-2026. このノートブックとそのソースコードは、MITライセンスの条件の下でリリースされています。